# Paper Experiment: Audio Digits Recognition (Part 1: Training)

This notebook implements steps 1, 2, and 3:
1. Download audio dataset.
2. Convert each audio sample to spectrogram according to paper specifications.
3. Train the model.

Finally, exports a zip archive for offline post-training quantization (PTQ).

**Before starting:** Under Runtime, select Change runtime type -> GPU, then run cells sequentially.

## Cell 1: Environment & GPU verification
No installation needed. Print library versions to document experiment configuration.

In [ ]:
import numpy, scipy, soundfile
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('NumPy', numpy.__version__, '| SciPy', scipy.__version__, '| SoundFile', soundfile.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 2: Download Speech Commands Dataset
The paper uses Speech Commands v0.01 dataset (~1.4 GB). Downloaded from Google storage.

In [ ]:
!wget -q --show-progress http://download.tensorflow.org/data/speech_commands_v0.01.tar.gz
!mkdir -p speech
!tar -xzf speech_commands_v0.01.tar.gz -C speech
!ls speech

## Cell 3: Audio to Spectrogram Preprocessing
Paper preprocessing pipeline (Section 3.2):
- Downsample audio from 16000 Hz to 8000 Hz.
- Segment 1-second audio into 39 frames, each 30 ms (240 samples) with 5 ms overlap (40 samples).
- Compute 256-point FFT per frame and retain the first 128 magnitude bins.
- Average every 4 consecutive bins to yield 32 frequency bands per frame.
- Apply logarithmic scaling: 20 * log10(magnitude).

Output shape: 32 x 39 spectrogram.

In [ ]:
import os, glob
import numpy as np
import soundfile as sf
from scipy.signal import decimate
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DATA_DIR = 'speech'
DIGITS = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
CLASSES = DIGITS + ['background', 'unknown']   # 12 classes as in the paper
FS = 8000      # Target sampling frequency (Hz)
FRAME = 240    # Frame length (30 ms at 8 kHz)
HOP = 200      # Hop length (240 - 40 overlap)
NFFT = 256

def load_clip(path):
    x, sr = sf.read(path, dtype='float32')
    x = decimate(x, 2)                         # 16kHz -> 8kHz
    x = np.pad(x, (0, max(0, FS - len(x))))[:FS]  # Pad or trim to exactly 1 second
    return x

def spectrogram(x):
    n_frames = 1 + (len(x) - FRAME) // HOP     # = 39
    frames = np.stack([x[i * HOP:i * HOP + FRAME] for i in range(n_frames)])
    mag = np.abs(np.fft.rfft(frames, n=NFFT, axis=1))[:, :128]
    mag = mag.reshape(n_frames, 32, 4).mean(axis=2)   # Average every 4 bins -> 32 bands
    spec = 20 * np.log10(mag + 1e-6)
    return spec.T.astype('float32')            # Output shape: (32, 39)

# Quick verification on sample word 'five'
f = sorted(glob.glob(DATA_DIR + '/five/*.wav'))[0]
x = load_clip(f)
s = spectrogram(x)
print('Spectrogram shape:', s.shape)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(x); ax[0].set_title('Waveform (five)')
ax[1].imshow(s, origin='lower', aspect='auto'); ax[1].set_title('spectrogram')
plt.show()

## Cell 4: Dataset Preparation
- Digits 0 to 9 each map to their respective class.
- Any other speech word is assigned to 'unknown' class.
- 'background' class is created by slicing 1-second noise clips from _background_noise_ folder.
- Split into train / val / test according to official validation and testing list files.



In [ ]:
val_set = set(open(DATA_DIR + '/validation_list.txt').read().split())
test_set = set(open(DATA_DIR + '/testing_list.txt').read().split())

X = {'train': [], 'val': [], 'test': []}
Y = {'train': [], 'val': [], 'test': []}

words = [d for d in sorted(os.listdir(DATA_DIR))
         if os.path.isdir(DATA_DIR + '/' + d) and not d.startswith('_')]
for w in words:
    label = CLASSES.index(w) if w in DIGITS else CLASSES.index('unknown')
    for f in tqdm(sorted(glob.glob(DATA_DIR + '/' + w + '/*.wav')), desc=w):
        rel = w + '/' + os.path.basename(f)
        split = 'test' if rel in test_set else ('val' if rel in val_set else 'train')
        X[split].append(spectrogram(load_clip(f)))
        Y[split].append(label)

# background class: 1-second random slices from noise files
rng = np.random.default_rng(0)
noise_files = glob.glob(DATA_DIR + '/_background_noise_/*.wav')
noise = [sf.read(p, dtype='float32')[0] for p in noise_files]
for split, n in [('train', 400), ('val', 50), ('test', 50)]:
    for _ in range(n):
        a = noise[rng.integers(len(noise))]
        i = rng.integers(len(a) - 16000)
        seg = decimate(a[i:i + 16000], 2)[:FS]
        X[split].append(spectrogram(seg))
        Y[split].append(CLASSES.index('background'))

for k in X:
    X[k] = np.array(X[k])[..., None]   # Shape: (N, 32, 39, 1)
    Y[k] = np.array(Y[k])
    print(k, X[k].shape, 'Samples per class:', np.bincount(Y[k], minlength=12))

## Cell 5: Normalization
Spectrogram dB values are standardized by subtracting mean and dividing by std (fitted on training data only).

Note: Standardization accelerates CNN training stability.

In [ ]:
MEAN = float(X['train'].mean())
STD = float(X['train'].std())
print('mean =', MEAN, ' std =', STD)
Xn = {k: ((X[k] - MEAN) / STD).astype('float32') for k in X}

## Cell 6: CNN Model Architecture
Architecture specified in paper (Section 3.3 and Table 2):
- 3 Conv2D layers with 12, 24, and 48 filters, 3x3 kernel size.
- Each Conv2D followed by ReLU activation and MaxPooling2D.
- 1 Fully-Connected (Dense) layer with 12 outputs and Softmax.
- Dropout regularizers between layers.

Pool3 parameters selected to match Table 2 output tensor (8x6x48 = 2304 features) resulting in exactly 40,812 parameters (Table 1).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model():
    return keras.Sequential([
        layers.Input(shape=(32, 39, 1)),
        layers.Conv2D(12, 3, padding='same', activation='relu', name='conv1'),
        layers.MaxPooling2D(2, padding='same'),
        layers.Dropout(0.2),
        layers.Conv2D(24, 3, padding='same', activation='relu', name='conv2'),
        layers.MaxPooling2D(2, padding='same'),
        layers.Dropout(0.2),
        layers.Conv2D(48, 3, padding='same', activation='relu', name='conv3'),
        layers.MaxPooling2D((1, 5), strides=(1, 1)),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(12, name='fc'),   # Raw logits (loss computes softmax internally)
    ])

model = build_model()
model.summary()
assert model.count_params() == 40812, 'Parameter count does not match paper specification (40,812)'

## Cell 7: Model Training
Training config: Adam optimizer, Categorical Crossentropy, 25 epochs.

In [ ]:
model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
history = model.fit(Xn['train'], Y['train'],
                    validation_data=(Xn['val'], Y['val']),
                    epochs=25, batch_size=256, verbose=2)

## Cell 8: Floating-point Baseline Evaluation
Paper reports ~86% accuracy on test partition. Evaluating baseline accuracy.

In [ ]:
from sklearn.metrics import confusion_matrix
logits = model.predict(Xn['test'], verbose=0)
pred = logits.argmax(axis=1)
acc = float((pred == Y['test']).mean())
print('Test set size:', len(Y['test']))
print('Float32 model accuracy: %.2f%%' % (acc * 100))
cm = confusion_matrix(Y['test'], pred)
plt.figure(figsize=(7, 6)); plt.imshow(cm); plt.colorbar()
plt.xticks(range(12), CLASSES, rotation=90); plt.yticks(range(12), CLASSES)
plt.title('confusion matrix'); plt.show()

## Cell 9: Export Artifacts for Embedded PTQ
- Model weights (npz): layer weights and biases for PTQ.
- Calibration set: 500 samples from training set (Section 4.2).
- Full test set.
- Metadata json file containing mean, std, baseline accuracy, and specs.

All packaged into an export zip file.

In [ ]:
import json, shutil
os.makedirs('export', exist_ok=True)

W = {}
for name in ['conv1', 'conv2', 'conv3', 'fc']:
    w, b = model.get_layer(name).get_weights()
    W[name + '_w'] = w; W[name + '_b'] = b
np.savez('export/weights_float32.npz', **W)
model.save('export/model_float32.keras')

idx = np.random.default_rng(1).choice(len(Xn['train']), 500, replace=False)
np.savez_compressed('export/calibration.npz', x=Xn['train'][idx], y=Y['train'][idx])
np.savez_compressed('export/test.npz', x=Xn['test'], y=Y['test'])

info = {'mean': MEAN, 'std': STD, 'test_accuracy_float32': acc,
        'n_test': int(len(Y['test'])), 'classes': CLASSES,
        'tensorflow': tf.__version__, 'numpy': numpy.__version__}
json.dump(info, open('export/info.json', 'w'), indent=2)

shutil.make_archive('paper_export', 'zip', 'export')
print(os.listdir('export'))
from google.colab import files
files.download('paper_export.zip')